# Feedlot Simulator V2 · Pilot dan Produksi

Data sintetis untuk portofolio. Kalender bisnis Edmonton, 1 Januari 2024–31 Desember 2025. Placement dihitung ketika ternak baru masuk selama periode tersebut.

1. Pilih **Runtime → Change runtime type → Hardware accelerator → None** dan sambungkan Colab.
2. Jalankan **Check Environment**, **Get Verified Source**, **Install Dependencies**, lalu **Connect Google Drive**.
3. Jalankan **Select Simulation Size & Preflight**. Pilot 1.500 menunggu bukti holdout Stage 5. Produksi 15.000 memerlukan pilot yang lulus, kapasitas runtime, dan persetujuan terpisah.
4. Setelah gerbang selesai, jalankan **Run Simulation**, **Validate Results**, dan **Save Results** secara berurutan.

Notebook dapat dibuka dan diperiksa tanpa menjalankan simulasi. [Panduan kesiapan dan pemulihan](https://github.com/wijaya-rama/portfolio-feedlot-management/blob/stage7-final-dataset-readiness/simulator_v2/stage6/colab/README.md).

## Check Environment
Periksa CPU, RAM, disk kosong dan versi Python sebelum menyiapkan sumber.

In [ ]:
# @title Check Environment
from datetime import date, datetime, timedelta, timezone
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
import uuid
from zoneinfo import ZoneInfo

REPOSITORY = "https://github.com/wijaya-rama/portfolio-feedlot-management.git"
REVIEWED_SHA = "41e1bbb963a0a08b95dd72c14e5281ac5ebea86c"
SEED = 42
GIB = 1024**3
PINS = ("numpy==2.3.5", "pyarrow==19.0.1", "tzdata==2025.2")
CONFIG_DIR = "simulator_v2/stage6/configs"
HEALTH_VERSION = "stage6b1-baseline-health.1"
SUPPLY_POLICY = "material_specific"
DATA_REPORTS = {
    "config.json", "diagnostics.json", "monthly_summary.json", "validation.json", "weather.json",
}
REQUIRED_GATES = {
    "balanced_paired_entries", "balanced_two_line_journal", "biological_reserve_balance",
    "financial_close_publication", "warehouse_replay_publication", "warehouse_af_value",
    "warehouse_dm", "stock_equation", "period_end_stock_reporting", "final_simulation_stock_reporting",
    "snapshot_publication", "stable_original_snapshot", "stable_restated_snapshot",
    "exact_placements", "lot_segregation", "baseline_operating_capacity", "hospital_headroom",
}
# Batas awal untuk ruang kerja, staging SQLite, dan salinan arsip. Satuan GiB.
RESOURCE_FLOORS = {1500: (8, 20, 12), 15000: (24, 120, 85)}


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def read_json(path):
    return json.loads(path.read_bytes())


def sha256_path(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def available_ram():
    line = next(row for row in Path("/proc/meminfo").read_text().splitlines()
                if row.startswith("MemAvailable:"))
    available = int(line.split()[1]) * 1024
    for limit_path, current_path in (
        ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
        ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
    ):
        limit_path, current_path = Path(limit_path), Path(current_path)
        if limit_path.is_file() and current_path.is_file():
            limit = limit_path.read_text().strip()
            if limit.isdecimal():
                available = min(available, max(0, int(limit) - int(current_path.read_text())))
    return available


def environment(root):
    return {
        "python": platform.python_version(),
        "system": platform.system(),
        "architecture": platform.machine(),
        "logical_cpus": len(os.sched_getaffinity(0)),
        "ram_available_bytes": available_ram(),
        "local_disk_free_bytes": shutil.disk_usage(root).free,
    }


def display_environment(info):
    print(f"Python {info['python']} | {info['system']} {info['architecture']}")
    print(f"CPU logis  {info['logical_cpus']} | RAM tersedia  {info['ram_available_bytes'] / GIB:.2f} GiB")
    print(f"Disk lokal kosong  {info['local_disk_free_bytes'] / GIB:.2f} GiB")


def check_runtime(root):
    require(platform.system() == "Linux" and platform.machine() == "x86_64",
            "Pilih runtime Linux x86_64 di Google Colab.")
    require(platform.python_implementation() == "CPython" and (3, 11) <= sys.version_info[:2] <= (3, 13),
            "Versi runtime perlu CPython 3.11–3.13. Pilih versi runtime Colab yang kompatibel.")
    require(not any(Path("/dev").glob("nvidia[0-9]*")) and not any(Path("/dev").glob("accel*"))
            and not any(os.environ.get(key) for key in ("COLAB_TPU_ADDR", "TPU_NAME", "TPU_WORKER_HOSTNAMES")),
            "Pilih Hardware accelerator None (CPU).")
    require(shutil.which("git") and shutil.disk_usage(root).free >= 2 * GIB,
            "Git dan setidaknya 2 GiB disk lokal diperlukan untuk menyiapkan runtime.")


def runner_env():
    env = os.environ.copy()
    for key in ("PYTHONPATH", "PYTHONHOME"):
        env.pop(key, None)
    env.update(PYTHONNOUSERSITE="1", PYTHONDONTWRITEBYTECODE="1", PYTHONHASHSEED="0", PYTHONTZPATH="")
    return env


LOCAL_ROOT = Path("/content")
require(LOCAL_ROOT.is_dir(), "Buka notebook di Google Colab agar /content tersedia.")
display_environment(environment(LOCAL_ROOT))
check_runtime(LOCAL_ROOT)
SESSION = Path(tempfile.mkdtemp(prefix="feedlot-stage7-", dir=LOCAL_ROOT))
SOURCE = SESSION / "reviewed-source"
PYTHON = SESSION / "venv" / "bin" / "python"
ATTEMPT = None
PHASES = ("operations", "finance")

## Get Verified Source
Ambil commit review publik dari GitHub dan periksa SHA checkout. Langkah ini tidak memerlukan kredensial GitHub.

In [ ]:
# @title Get Verified Source
def git(source, *args):
    return subprocess.check_output(
        ["git", "-C", str(source), *args], text=True,
        env={**runner_env(), "GIT_TERMINAL_PROMPT": "0"},
    ).strip()


def verify_source(source):
    require(git(source, "rev-parse", "HEAD") == REVIEWED_SHA,
            "SHA kode simulator berbeda dari commit review.")
    require(not git(source, "status", "--porcelain", "--untracked-files=all"),
            "Checkout simulator berubah. Mulai sesi baru untuk mengambil kode bersih.")


def get_source(source):
    if not source.joinpath(".git").is_dir():
        source.mkdir(exist_ok=True)
        git(source, "init", "--quiet")
    if subprocess.run(["git", "-C", str(source), "cat-file", "-e", f"{REVIEWED_SHA}^{{commit}}"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
        git(source, "fetch", "--no-tags", "--depth=1", REPOSITORY, REVIEWED_SHA)
        require(git(source, "rev-parse", "FETCH_HEAD") == REVIEWED_SHA, "SHA dari GitHub tidak cocok.")
    if subprocess.run(["git", "-C", str(source), "rev-parse", "--verify", "HEAD"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
        git(source, "checkout", "--detach", "--quiet", REVIEWED_SHA)
    verify_source(source)


check_runtime(SESSION)
get_source(SOURCE)
print(f"Commit review terverifikasi  {REVIEWED_SHA}")

## Install Dependencies
Buat lingkungan Python lokal terisolasi. Verifikasi versi paket, codec Zstandard, zona waktu Edmonton
dan impor runner sebelum menjalankan simulasi.

In [ ]:
# @title Install Dependencies
def check_dependencies(python, source):
    script = """
from importlib.metadata import version
import sys
from zoneinfo import ZoneInfo
import numpy
import pyarrow as pa
import pyarrow.parquet
from simulator_v2.stage6.__main__ import build
assert all(version(name) == expected for name, expected in (pin.split('==') for pin in sys.argv[1:]))
assert pa.Codec.is_available('zstd')
assert ZoneInfo('America/Edmonton')
"""
    subprocess.run([str(python), "-c", script, *PINS], cwd=source, env=runner_env(), check=True)


def install_dependencies(python, source):
    subprocess.run([sys.executable, "-m", "venv", str(python.parents[1])], check=True)
    subprocess.run([str(python), "-m", "pip", "install", "--disable-pip-version-check",
                    "--no-cache-dir", "--only-binary=:all:", *PINS], check=True)
    subprocess.run([str(python), "-m", "pip", "check"], check=True)
    check_dependencies(python, source)


verify_source(SOURCE)
install_dependencies(PYTHON, SOURCE)
print("Dependensi runtime siap ", ", ".join(PINS))

## Connect Google Drive
Izinkan Colab mengakses Google Drive saat diminta. Simulator bekerja di disk lokal Colab.
Drive menyimpan status, laporan, dan satu arsip dataset setelah validasi.

In [ ]:
# @title Connect Google Drive
from google.colab import drive

MOUNT = LOCAL_ROOT / "drive"
drive.mount(str(MOUNT))
require(MOUNT.joinpath("MyDrive").is_dir(), "Google Drive belum terhubung.")
DRIVE_ROOT = MOUNT / "MyDrive" / "Feedlot_V2_Releases"
DRIVE_ROOT.mkdir(exist_ok=True)
print("Folder hasil di Drive: My Drive / Feedlot_V2_Releases")

## Select Simulation Size & Preflight

Preflight memeriksa jadwal, kandang, peralatan, dan pasokan 56 hari dengan model jadwal serta transaksi gudang berukuran kecil. Pilih **1500** untuk pilot atau **15000** untuk konfigurasi produksi.

In [ ]:
# @title Select Simulation Size & Preflight
Placements = "1500" # @param ["1500", "15000"]


def config_for(source, size):
    require(size in RESOURCE_FLOORS, "Pilih 1.500 atau 15.000.")
    name = "pilot" if size == 1500 else "production"
    path = source / CONFIG_DIR / f"stage7-{name}-{size}.json"
    config = read_json(path)
    require(config["version"] == f"stage7-{name}-{size}.1" and config["seed"] == SEED
            and (config["start"], config["end"]) == ("2024-01-01", "2025-12-31")
            and config["required_placements"] == sum(item["head"] for item in config["placements"]) == size
            and config["lot_segregation"] is True and config["health"]["scenario_version"] == HEALTH_VERSION,
            "Konfigurasi rilis tidak sesuai ukuran, kalender, seed, atau kebijakan lot.")
    return config


def verify_audit(validation):
    require(validation.get("mode") == "full" and validation.get("complete_independent_audit") is True,
            "Audit independen penuh belum lulus.")
    gates = validation["hard_gates"]
    require(REQUIRED_GATES <= gates.keys(), "Laporan audit kehilangan gerbang wajib.")
    require(all(row["passed"] is True for row in gates.values())
            and all(gates[name]["assertions"] > 0 for name in REQUIRED_GATES),
            "Audit memuat gerbang gagal atau gerbang wajib yang belum dijalankan.")
    for category in ("movement", "financial"):
        checks = validation[category]
        require(checks and all(check["failed_count"] == 0 for check in checks),
                f"Kontrol {category} kosong atau gagal.")
    return len(gates)


def check_pilot_result(root):
    candidates = []
    for path in root.glob("placements-1500-*/status.json"):
        try:
            status = read_json(path)
            if (status["status"], status["validation_status"], status["code_sha"], status["placements"],
                    status["config_version"], status["seed"]) != (
                    "completed", "passed", REVIEWED_SHA, 1500, "stage7-pilot-1500.1", SEED):
                continue
            require(status["complete_independent_audit"] is True and status["head"]["placements"] == 1500
                    and status["financial_closes"] == 48, "Bukti pilot belum lengkap.")
            for name in ("validation.json", "performance.json", "run_manifest.json", "config.json"):
                require(sha256_path(path.parent / name) == status["saved_sha256"][name],
                        f"Bukti pilot berubah ({name}).")
            verify_audit(read_json(path.parent / "validation.json"))
            require(read_json(path.parent / "performance.json") == status["performance"]
                    and read_json(path.parent / "config.json") == config_for(SOURCE, 1500)
                    and read_json(path.parent / "run_manifest.json")["code_checkout_commit_at_build"] == REVIEWED_SHA,
                    "Metrik, konfigurasi, atau commit pilot berbeda.")
            package = path.parent / "dataset.tar"
            require(package.stat().st_size == status["package_bytes"]
                    and sha256_path(package) == status["saved_sha256"]["dataset.tar"],
                    "Paket pilot berubah atau belum selesai disalin.")
            candidates.append(status)
        except (OSError, ValueError, KeyError, TypeError, RuntimeError):
            continue
    require(candidates, "Produksi menunggu satu pilot 1.500 yang selesai dan terverifikasi di Drive.")
    return max(candidates, key=lambda item: item["updated_at"])


def require_resources(size, local, drive_root, previous=None, checkpoint_credit=0):
    ram, local_disk, durable_disk = (amount * GIB for amount in RESOURCE_FLOORS[size])
    if previous:
        scale = size / previous["placements"]
        measured = previous["performance"]
        projected = previous["package_bytes"] * scale
        ram = max(ram, measured["peak_rss_kib_linux"] * 1024 * scale * 1.25)
        local_disk = max(local_disk, projected * 5 + GIB)
        saved = sum(item["bytes"] for item in previous["checkpoints"].values())
        durable_disk = max(durable_disk, (previous["package_bytes"] + saved) * scale * 1.25)
    for label, free, limit in (
        ("RAM tersedia", available_ram(), ram),
        ("Disk lokal kosong", shutil.disk_usage(local).free, local_disk),
        ("Ruang Drive yang dilaporkan", shutil.disk_usage(drive_root).free + checkpoint_credit, durable_disk),
    ):
        print(f"{label} {free / GIB:.2f} GiB | batas {limit / GIB:.2f} GiB")
        require(free >= limit, f"{label} belum cukup untuk {size} placement.")


SIZE = int(Placements)
verify_source(SOURCE)
check_dependencies(PYTHON, SOURCE)
SELECTED_CONFIG = config_for(SOURCE, SIZE)
result = subprocess.check_output([str(PYTHON), "-m", "simulator_v2.stage6.release_preflight", str(SIZE)],
                                 cwd=SOURCE, env=runner_env(), text=True)
PREFLIGHT = json.loads(result)
peak = PREFLIGHT["sale_delay_days"]["21"]
print(f"{SIZE:,} placement | {PREFLIGHT['carry_in']:,} carry-in | puncak rencana {peak['concurrent_head']:,} ekor"
      .replace(",", "."))
print(f"Cadangan {peak['production_spare_pens']} pen produksi dan {peak['receiving_spare_pens']} pen receiving.")
print("Pasokan per bahan dan FIFO lulus dalam uji permintaan 56 hari untuk lima seed.")
print("Lanjutkan setelah gerbang estimator, persetujuan, dan kapasitas runtime terpenuhi.")

## Run Simulation

Biarkan **Mulai_atau_lanjutkan** kosong untuk membaca panduan. Untuk menjalankan, centang kotak dan ketik frasa yang ditampilkan. **Attempt_ID** kosong membuat percobaan baru. Untuk pemulihan, isi dengan nama folder Drive percobaan sebelumnya yang diawali `placements-1500-` atau `placements-15000-`.

Checkpoint operasi dan keuangan disimpan ke Drive setelah masing-masing fase selesai. Jika runtime berhenti di tengah fase, fase tersebut diulang dari checkpoint terakhir yang terverifikasi. Gunakan checkpoint dari percobaan Anda sendiri.

In [ ]:
# @title Run Simulation
Mulai_atau_lanjutkan = False # @param {type:"boolean"}
Gerbang_estimator_selesai = False # @param {type:"boolean"}
Saya_menyetujui_pilot_1500 = False # @param {type:"boolean"}
Saya_menyetujui_produksi_15000 = False # @param {type:"boolean"}
Attempt_ID = "" # @param {type:"string"}


def write_status(attempt, state, **fields):
    updated = {**attempt["status"], **fields, "status": state,
               "updated_at": datetime.now(timezone.utc).isoformat()}
    path = attempt["durable"] / "status.json"
    temporary = path.with_name(f".status-{uuid.uuid4().hex}.partial")
    temporary.write_text(json.dumps(updated, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)
    attempt["status"] = updated


def record_failure(attempt, state, error, **fields):
    try:
        write_status(attempt, state, error=f"{type(error).__name__}: {error}", **fields)
        log = attempt["work"] / "runner.log"
        if log.is_file():
            copy_verified(log, attempt["durable"] / "runner.log")
    except (OSError, RuntimeError) as save_error:
        print(f"Pencatatan status di Drive terhenti ({save_error}). Buka folder percobaan untuk memeriksa.")


def copy_verified(source, destination, expected=None):
    expected = expected or sha256_path(source)
    size = source.stat().st_size
    if destination.is_file() and destination.stat().st_size == size and sha256_path(destination) == expected:
        return expected
    temporary = destination.with_name(f".{destination.name}-{uuid.uuid4().hex}.partial")
    try:
        shutil.copyfile(source, temporary)
        require(temporary.stat().st_size == size and sha256_path(temporary) == expected,
                f"Salinan gagal diverifikasi ({destination.name}).")
        temporary.replace(destination)
        return expected
    finally:
        temporary.unlink(missing_ok=True)


def make_package(directory, archive):
    files = sorted(path for path in directory.rglob("*") if path.is_file())
    reserve = sum(path.stat().st_size + 4096 for path in files) + 100 * 1024
    require(shutil.disk_usage(archive.parent).free >= reserve + GIB // 2,
            "Disk lokal belum cukup untuk membuat arsip.")
    temporary = archive.with_name(f".{archive.name}-{uuid.uuid4().hex}.partial")
    try:
        with tarfile.open(temporary, "w", format=tarfile.PAX_FORMAT) as output:
            for path in files:
                output.add(path, arcname=(Path(directory.name) / path.relative_to(directory)).as_posix(),
                           recursive=False)
        temporary.replace(archive)
    finally:
        temporary.unlink(missing_ok=True)


def save_checkpoint(attempt, phase):
    work, durable = attempt["work"], attempt["durable"]
    phase_path = work / "checkpoints" / phase
    require((phase_path / "checkpoint.json").is_file(), f"Marker checkpoint belum tersedia ({phase}).")
    scratch = work / f"{phase}.tar"
    make_package(phase_path, scratch)
    try:
        require(shutil.disk_usage(durable).free >= scratch.stat().st_size + 100 * 1024**2,
                "Ruang Drive belum cukup untuk menyimpan checkpoint.")
        filename = f"checkpoint-{phase}-{uuid.uuid4().hex[:12]}.tar"
        phase_info = {"file": filename, "sha256": copy_verified(scratch, durable / filename),
                      "bytes": scratch.stat().st_size, "checkpoint_sha256": sha256_path(phase_path / "checkpoint.json")}
        completed = {**attempt["status"]["checkpoints"], phase: phase_info}
        write_status(attempt, "checkpoint_saved", checkpoints=completed, last_verified_checkpoint=phase)
        print(f"Checkpoint Drive terverifikasi {phase} | {phase_info['bytes'] / GIB:.2f} GiB", flush=True)
    finally:
        scratch.unlink(missing_ok=True)


def restore_checkpoint(attempt, phase, info):
    work, durable = attempt["work"], attempt["durable"]
    require(re.fullmatch(rf"checkpoint-{phase}-[0-9a-f]{{12}}\.tar", info["file"]),
            "Nama checkpoint Drive tidak valid.")
    target = work / "checkpoints" / phase
    if target.is_dir():
        require(sha256_path(target / "checkpoint.json") == info["checkpoint_sha256"],
                "Checkpoint lokal berbeda dari bukti Drive. Sambungkan runtime baru untuk memulihkan.")
        return
    archive = durable / info["file"]
    require(archive.stat().st_size == info["bytes"] and sha256_path(archive) == info["sha256"],
            f"SHA-256 checkpoint Drive tidak cocok ({phase}).")
    with tempfile.TemporaryDirectory(prefix="restore-", dir=target.parent) as temporary:
        with tarfile.open(archive, "r:") as stream:
            members = stream.getmembers()
            names = [item.name for item in members]
            require(len(set(names)) == len(names) and all(
                item.isfile() and item.name.startswith(phase + "/") and ".." not in Path(item.name).parts
                and not Path(item.name).is_absolute() for item in members
            ), "Isi arsip checkpoint tidak sesuai.")
            stream.extractall(temporary, members=members, filter="data")
        reconstructed = Path(temporary) / phase
        require(sha256_path(reconstructed / "checkpoint.json") == info["checkpoint_sha256"],
                "Marker checkpoint hasil pemulihan berbeda dari bukti Drive.")
        reconstructed.replace(target)


def open_attempt(config, size, attempt_id):
    resumed = bool(attempt_id)
    if not resumed:
        attempt_id = f"placements-{size}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:8]}"
    work, durable = SESSION / attempt_id, DRIVE_ROOT / attempt_id
    if resumed:
        state = read_json(durable / "status.json")
        require((state["attempt_id"], state["code_sha"], state["placements"], state["config_version"], state["seed"])
                == (attempt_id, REVIEWED_SHA, size, config["version"], SEED)
                and state["status"] != "completed" and read_json(durable / "selected-config.json") == config,
                "Percobaan selesai atau versi dan konfigurasi berbeda.")
    else:
        state = {"attempt_id": attempt_id, "code_sha": REVIEWED_SHA, "placements": size, "seed": SEED,
                 "config_version": config["version"], "health_version": HEALTH_VERSION, "supply_policy": SUPPLY_POLICY,
                 "started_at": datetime.now(timezone.utc).isoformat(), "resources": environment(SESSION),
                 "dependencies": list(PINS), "validation_status": "pending", "complete_independent_audit": False,
                 "checkpoints": {}, "worker_seconds": 0.0}
    work.mkdir(exist_ok=True)
    (work / "checkpoints").mkdir(exist_ok=True)
    local_config = work / "selected-config.json"
    local_config.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
    attempt = {"work": work, "durable": durable, "config": config, "status": state}
    if not resumed:
        durable.mkdir()
        copy_verified(local_config, durable / local_config.name)
        write_status(attempt, "prepared")
    return attempt


def run_worker(attempt, phase=None):
    work = attempt["work"]
    command = [str(PYTHON), "-u", "-m", "simulator_v2.stage6", "--config", str(work / "selected-config.json"),
               "--output", str(work / "dataset"), "--checkpoint-dir", str(work / "checkpoints")]
    label = phase or "audit_and_publish"
    if phase:
        command.extend(["--stop-after", phase])
    log = work / "runner.log"
    started = time.monotonic()
    write_status(attempt, "running", phase=label, command=command, error=None)
    print(f"Fase {label} sedang berjalan.", flush=True)
    with log.open("ab") as output:
        output.write(f"\n--- {label} ---\n".encode())
        output.flush()
        with subprocess.Popen(command, cwd=SOURCE, env=runner_env(), stdout=output, stderr=subprocess.STDOUT) as worker:
            next_report = 60
            try:
                while True:
                    try:
                        code = worker.wait(timeout=10)
                        break
                    except subprocess.TimeoutExpired:
                        elapsed = time.monotonic() - started
                        require(shutil.disk_usage(work).free >= GIB // 2, "Disk lokal mendekati penuh.")
                        if elapsed >= next_report:
                            print(f"{label} masih berjalan, {elapsed / 60:.0f} menit.", flush=True)
                            next_report += 60
            finally:
                if worker.poll() is None:
                    worker.terminate()
                    try:
                        worker.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        worker.kill()
    if code:
        with log.open("rb") as stream:
            stream.seek(max(0, log.stat().st_size - 4000))
            print(stream.read().decode(errors="replace"))
    require(code == 0, f"Fase {label} berhenti dengan kode {code}. Buka runner.log.")
    seconds = attempt["status"]["worker_seconds"] + (time.monotonic() - started)
    write_status(attempt, "phase_done_locally", worker_seconds=round(seconds, 3))


def run_simulation(attempt):
    print(f"Nama folder Drive untuk pemulihan {attempt['status']['attempt_id']}", flush=True)
    try:
        for phase in PHASES:
            shown = attempt["status"]["checkpoints"].get(phase)
            if shown:
                restore_checkpoint(attempt, phase, shown)
            else:
                run_worker(attempt, phase)
                save_checkpoint(attempt, phase)
        if not (attempt["work"] / "dataset").exists():
            run_worker(attempt)
        write_status(attempt, "awaiting_validation", validation_status="unverified",
                     complete_independent_audit=False, runner_wall_seconds=attempt["status"]["worker_seconds"])
        print("Runner selesai. Jalankan Validate Results.")
    except BaseException as error:
        record_failure(attempt, "interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                       error, validation_status="unverified", complete_independent_audit=False)
        raise


if Mulai_atau_lanjutkan:
    verify_source(SOURCE)
    check_dependencies(PYTHON, SOURCE)
    require(PREFLIGHT["version"] == SELECTED_CONFIG["version"], "Jalankan preflight untuk konfigurasi ini.")
    require(Gerbang_estimator_selesai, "Gerbang estimator Stage 5 menunggu bukti holdout sesuai protokol.")
    if SIZE == 1500:
        require(Saya_menyetujui_pilot_1500, "Pilot 1.500 memerlukan persetujuan.")
        previous = None
    else:
        require(Saya_menyetujui_produksi_15000, "Produksi 15.000 memerlukan persetujuan terpisah.")
        previous = check_pilot_result(DRIVE_ROOT)
    requested = Attempt_ID.strip()
    credit = 0
    if requested:
        require(re.fullmatch(rf"placements-{SIZE}-[0-9]{{8}}-[0-9]{{6}}-[0-9a-f]{{8}}", requested),
                "Nama folder percobaan tidak valid.")
        existing = read_json(DRIVE_ROOT / requested / "status.json")
        credit = sum(row["bytes"] for row in existing.get("checkpoints", {}).values())
    require_resources(SIZE, SESSION, DRIVE_ROOT, previous, credit)
    action = "LANJUT" if requested else "MULAI"
    phrase = f"{action} {SIZE}"
    if input(f"Ketik {phrase} lalu tekan Enter. ").strip() == phrase:
        ATTEMPT = open_attempt(SELECTED_CONFIG, SIZE, requested)
        run_simulation(ATTEMPT)
    else:
        print("Permintaan dibatalkan.")
else:
    print("Belum ada simulasi yang dijalankan.")

## Validate Results
Runner melakukan audit penuh atas Parquet sebelum publikasi lokal. Sel ini memeriksa kelengkapan
audit, checksum partisi dan laporan, carry-in/carry-out, serta penutupan tiap bulan untuk kedua view.
Audit incremental mencakup sebagian pemeriksaan dan berstatus `complete_independent_audit=false`.

In [ ]:
# @title Validate Results
def validate_dataset(dataset, config):
    started = time.monotonic()
    manifest = read_json(dataset / "run_manifest.json")
    index = read_json(dataset / "hashes.json")
    require(manifest["code_checkout_commit_at_build"] == REVIEWED_SHA and manifest["synthetic"] is True
            and manifest["run_id"] == index["run_id"] and manifest["schema_version"] == index["schema_version"],
            "Manifest tidak cocok dengan kode review atau indeks hasil.")
    require(manifest["hashes_sha256"] == sha256_path(dataset / "hashes.json")
            and manifest["config_sha256"] == sha256_path(dataset / "config.json")
            and read_json(dataset / "config.json") == config,
            "Hash deterministik atau konfigurasi keluaran tidak cocok.")
    end = datetime.fromisoformat(manifest["effective_before"]).astimezone(ZoneInfo("America/Edmonton"))
    cutoff = datetime.fromisoformat(manifest["final_knowledge_cutoff"])
    require(end.date() == date.fromisoformat(config["end"]) + timedelta(days=1)
            and cutoff - end == timedelta(days=max(config["shipment"]["settlement_lag_days"]) + 4),
            "Horizon efektif atau cutoff penutupan tidak cocok.")

    partitions, reports = index["partitions"], index["artifacts"]
    require(bool(partitions) and set(reports) == DATA_REPORTS, "Indeks partisi atau laporan belum lengkap.")
    actual = {path.relative_to(dataset).as_posix() for path in dataset.rglob("*.parquet")}
    require(actual == partitions.keys(), "Inventaris Parquet berbeda dari indeks deterministik.")
    for name, row in partitions.items():
        path = dataset / name
        require(path.stat().st_size == row["bytes"] and sha256_path(path) == row["sha256"],
                f"Ukuran atau SHA-256 partisi berbeda ({name}).")
    for name, digest in reports.items():
        require(sha256_path(dataset / name) == digest, f"SHA-256 laporan berbeda ({name}).")

    validation = read_json(dataset / "validation.json")
    gate_count = verify_audit(validation)
    expected_months = set()
    month = date.fromisoformat(config["start"]).replace(day=1)
    final_date = date.fromisoformat(config["end"])
    while month <= final_date:
        expected_months.add(month.isoformat()[:7])
        month = (month.replace(day=28) + timedelta(days=4)).replace(day=1)
    closes = read_json(dataset / "monthly_summary.json")
    for view in ("as_known", "restated"):
        months = [row["month"] for row in closes if row["view"] == view]
        require(len(months) == len(expected_months) and set(months) == expected_months,
                f"Penutupan bulanan tidak lengkap untuk {view}.")
    require(len(closes) == 2 * len(expected_months), "Ada view atau penutupan tambahan yang tidak diharapkan.")

    diagnostics = read_json(dataset / "diagnostics.json")
    head = diagnostics["head"]
    stock = diagnostics["stock_reporting"]
    require(head["placements"] == config["required_placements"]
            and head["opening"] == sum(row["head"] for row in config["carry_in"])
            and stock["final_simulation_date"] == config["end"]
            and head["carry_out"] == stock["final_simulation_soh"]
            and head["opening"] + head["placements"] == head["sale"] + head["death"] + head["carry_out"],
            "Carry-in, carry-out, atau rekonsiliasi jumlah ternak berbeda.")
    performance = read_json(dataset / "performance.json")
    for field in ("wall_seconds_before_atomic_rename", "cross_domain_validation_seconds", "peak_rss_kib_linux"):
        require(math.isfinite(performance[field]) and performance[field] > 0, f"Metrik {field} belum lengkap.")
    require(performance["parquet_bytes"] == sum(row["bytes"] for row in partitions.values())
            and performance["parquet_partitions"] == len(partitions), "Metrik Parquet berbeda dari inventaris.")
    return {"run_id": manifest["run_id"], "hashes_sha256": manifest["hashes_sha256"],
            "model_version": manifest["model_version"], "schema_version": manifest["schema_version"],
            "gate_count": gate_count, "financial_closes": len(closes), "head": head, "performance": performance,
            "validation_mode": "full", "complete_independent_audit": True, "validation_status": "passed",
            "artifact_verification_seconds": round(time.monotonic() - started, 3)}


def validate_attempt(attempt):
    require(attempt["status"]["status"] == "awaiting_validation", "Run belum siap divalidasi.")
    try:
        result = validate_dataset(attempt["work"] / "dataset", attempt["config"])
        write_status(attempt, "validated", **result)
        print(f"Audit penuh lulus, {result['gate_count']} kategori dan {result['financial_closes']} penutupan. Jalankan Save Results.")
    except BaseException as error:
        record_failure(attempt, "validation_failed", error, validation_status="failed",
                       complete_independent_audit=False)
        raise


if ATTEMPT and ATTEMPT["status"]["status"] == "awaiting_validation":
    validate_attempt(ATTEMPT)
else:
    print("Tidak ada run baru yang menunggu validasi.")

## Save Results

Drive menyimpan paket dataset dan ringkasan setelah audit penuh serta pemeriksaan hash lulus. Jika penyalinan berhenti, jalankan kembali **Save Results** pada runtime yang sama atau masukkan **Attempt_ID** bila runtime sudah hilang.

In [ ]:
# @title Save Results
def save_attempt(attempt):
    state = attempt["status"]
    require(state["status"] in {"validated", "save_failed"} and state["validation_status"] == "passed"
            and state["complete_independent_audit"] is True, "Penyimpanan menunggu audit penuh yang lulus.")
    started = time.monotonic()
    work, durable = attempt["work"], attempt["durable"]
    dataset, archive = work / "dataset", work / "dataset.tar"
    try:
        if not archive.is_file() or "local_archive_sha256" not in attempt:
            archive.unlink(missing_ok=True)
            make_package(dataset, archive)
            attempt["local_archive_sha256"] = sha256_path(archive)
        require(shutil.disk_usage(durable).free >= archive.stat().st_size + 100 * 1024**2,
                "Ruang Drive yang dilaporkan belum cukup untuk menyimpan paket.")
        saved = {}
        for name in ("config.json", "run_manifest.json", "hashes.json", "validation.json", "performance.json"):
            saved[name] = copy_verified(dataset / name, durable / name)
        saved["runner.log"] = copy_verified(work / "runner.log", durable / "runner.log")
        saved["dataset.tar"] = copy_verified(archive, durable / archive.name, attempt["local_archive_sha256"])
        write_status(attempt, "completed", saved_sha256=saved, package_bytes=archive.stat().st_size,
                     save_seconds=round(time.monotonic() - started, 3), error=None)
        print(f"Paket terverifikasi tersimpan di My Drive / Feedlot_V2_Releases / {durable.name}")
    except BaseException as error:
        record_failure(attempt, "save_failed", error)
        raise


if ATTEMPT and ATTEMPT["status"]["status"] in {"validated", "save_failed"}:
    save_attempt(ATTEMPT)
else:
    print("Paket baru akan disimpan sesudah audit penuh lulus.")

## View Summary
Periksa status akhir, durasi runner, waktu audit penuh, puncak RSS proses,
ukuran Parquet aktual dan lokasi paket tersimpan.

In [ ]:
# @title View Summary
def view_summary(attempt):
    status = read_json(attempt["durable"] / "status.json")
    print(f"Status tersimpan  {status['status']} | Audit  {status['validation_status']}")
    print(f"Placement diminta  {status['placements']} | Seed  {status['seed']} | Commit  {status['code_sha']}")
    if status.get("error"):
        print(status["error"])
    if status.get("validation_status") == "passed":
        measured = status["performance"]
        print("Pengukuran dari runtime CPU ini")
        print(f"Durasi runner  {status['runner_wall_seconds']:.2f} detik")
        print(f"Audit penuh  {measured['cross_domain_validation_seconds']:.2f} detik")
        print(f"Pemeriksaan integritas  {status['artifact_verification_seconds']:.2f} detik")
        print(f"Puncak RSS proses simulator  {measured['peak_rss_kib_linux'] * 1024 / GIB:.2f} GiB")
        for phase, measured_phase in measured["phases"].items():
            print(f"{phase} {measured_phase['seconds']:.1f} detik")
        print(f"Parquet  {measured['parquet_bytes'] / GIB:.2f} GiB | Penutupan  {status['financial_closes']}")
        print(f"Masuk aktual  {status['head']['placements']} | Carry-in  {status['head']['opening']} | Carry-out  {status['head']['carry_out']}")
    if status["status"] == "completed":
        print(f"Paket selesai  My Drive / Feedlot_V2_Releases / {attempt['durable'].name}/dataset.tar")
    else:
        print("Paket final belum terkonfirmasi di Drive.")


if ATTEMPT:
    view_summary(ATTEMPT)
else:
    print("Belum ada run pada sesi ini.")